Imports & Hardware Initialization

In [ ]:
from pynq import Overlay, MMIO
import matplotlib.pyplot as plt
import numpy as np
import math
import time

# ====================================
# 1. Initialization and Configuration
# ====================================
print("Loading overlay...")
overlay = Overlay("beamform_ver_1_wrapper.bit")
print("Overlay loaded successfully!")

# Base address assigned to your custom IP by Vivado
ip = MMIO(0x40000000, 0x1000)

# Brief pause to let the hardware memory map settle
time.sleep(0.5)

The Hardware Helper Functions


In [ ]:
# ====================================
# 2. Hardware Helper Functions
# ====================================
def float_to_sfix16_en12(val):
    """Converts Python float to 16-bit fixed-point for the FPGA"""
    max_val, min_val = (2**15 - 1) / 4096.0, -(2**15) / 4096.0
    val = max(min(val, max_val), min_val)
    int_val = int(round(val * 4096))
    if int_val < 0:
        int_val = (1 << 16) + int_val
    return int_val & 0xFFFF

def sfix16_en12_to_float(val):
    """Converts 16-bit fixed-point from the FPGA back to Python float"""
    val = val & 0xFFFF
    if val >= 32768:
        val -= 65536
    return val / 4096.0

def write_complex_input(base_addr, real_val, imag_val, strobe_addr):
    """Writes a complex number to the hardware and fires the strobe"""
    ip.write(base_addr, float_to_sfix16_en12(real_val))
    ip.write(base_addr + 0x04, float_to_sfix16_en12(imag_val))
    ip.write(strobe_addr, 1)

def read_complex_output():
    """Fires the output strobe and reads the computed result"""
    ip.write(0x148, 1)  # Output strobe
    return sfix16_en12_to_float(ip.read(0x140)), sfix16_en12_to_float(ip.read(0x144))

Test 1 - Spatial Sweep

In [ ]:

# ====================================
# 3. Spatial Sweep Execution
# ====================================
ip.write(0x004, 1) # Enable IP

# We will steer the hardware to 30 degrees
steer_angle_deg = 30.0
theta_rad = math.radians(steer_angle_deg)
ip.write(0x10C, float_to_sfix16_en12(theta_rad))

# Arrays to store our plotting data
angles_to_test = np.linspace(-90, 90, 181) # Test every degree from -90 to 90
magnitudes = []

print("Sweeping angles and querying the FPGA...")
for angle in angles_to_test:
    # 1. Calculate the phase shift for this specific arrival angle
    arrival_rad = math.radians(angle)
    phase_shift = math.pi * math.sin(arrival_rad)

    # 2. Generate the 4 incoming signals (Amplitude = 0.5)
    amplitude = 0.5
    signals = []
    for i in range(4):
        element_phase = -(i * phase_shift)
        real_part = amplitude * math.cos(element_phase)
        imag_part = amplitude * math.sin(element_phase)
        signals.append((real_part, imag_part))

    # 3. Send to hardware
    write_complex_input(0x100, signals[0][0], signals[0][1], 0x108) # In1
    write_complex_input(0x110, signals[1][0], signals[1][1], 0x118) # In2
    write_complex_input(0x120, signals[2][0], signals[2][1], 0x128) # In3
    write_complex_input(0x130, signals[3][0], signals[3][1], 0x138) # In4

    # 4. Read the output from the FPGA
    y_real, y_imag = read_complex_output()
    mag = math.sqrt(y_real**2 + y_imag**2)
    magnitudes.append(mag)

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(angles_to_test, magnitudes, linewidth=2, color='b')
plt.title(f"Hardware Beamformer Radiation Pattern (Steered to {steer_angle_deg}°)")
plt.xlabel("Angle of Arrival (Degrees)")
plt.ylabel("Output Magnitude")
plt.axvline(x=steer_angle_deg, color='r', linestyle='--', label=f'Steering Direction ({steer_angle_deg}°)')
plt.grid(True)
plt.legend()
plt.xlim(-90, 90)
plt.show()

Test 2 - Time-Domain Sine Wave Execution

In [ ]:
# ====================================
# 4. Time-Domain Sine Wave Execution
# ====================================
ip.write(0x004, 1) # Enable IP

# Steer the FPGA to 30 degrees
steer_angle_deg = 30.0
ip.write(0x10C, float_to_sfix16_en12(math.radians(steer_angle_deg)))

# Generate time steps for 2 full cycles of a sine wave
time_steps = np.linspace(0, 4 * math.pi, 150)
phase_shift = math.pi * math.sin(math.radians(steer_angle_deg))

# Lists to hold plotting data
in1_plot, in2_plot, in3_plot, in4_plot, out_plot = [], [], [], [], []

print("Running continuous sine wave through the FPGA fabric...")

for t in time_steps:
    # Calculate the instantaneous amplitude of the sine wave
    # We apply the physical delay (phase shift) to each element
    sig0_r = 0.5 * math.cos(t - 0 * phase_shift)
    sig0_i = 0.5 * math.sin(t - 0 * phase_shift)

    sig1_r = 0.5 * math.cos(t - 1 * phase_shift)
    sig1_i = 0.5 * math.sin(t - 1 * phase_shift)

    sig2_r = 0.5 * math.cos(t - 2 * phase_shift)
    sig2_i = 0.5 * math.sin(t - 2 * phase_shift)

    sig3_r = 0.5 * math.cos(t - 3 * phase_shift)
    sig3_i = 0.5 * math.sin(t - 3 * phase_shift)

    # Push to hardware
    write_complex_input(0x100, sig0_r, sig0_i, 0x108)
    write_complex_input(0x110, sig1_r, sig1_i, 0x118)
    write_complex_input(0x120, sig2_r, sig2_i, 0x128)
    write_complex_input(0x130, sig3_r, sig3_i, 0x138)

    # Read the real-time computed output from the fabric
    y_real, y_imag = read_complex_output()

    # Store the REAL parts to draw standard sine waves on the graph
    in1_plot.append(sig0_r)
    in2_plot.append(sig1_r)
    in3_plot.append(sig2_r)
    in4_plot.append(sig3_r)
    out_plot.append(y_real)

# Draw the Sine Wave Plot
plt.figure(figsize=(12, 6))

# Plot the 4 staggered input waves (dashed lines)
plt.plot(time_steps, in1_plot, linestyle='--', alpha=0.7, label='Antenna 1 Input')
plt.plot(time_steps, in2_plot, linestyle='--', alpha=0.7, label='Antenna 2 Input')
plt.plot(time_steps, in3_plot, linestyle='--', alpha=0.7, label='Antenna 3 Input')
plt.plot(time_steps, in4_plot, linestyle='--', alpha=0.7, label='Antenna 4 Input')

# Plot the massive output wave from the FPGA (solid bold line)
plt.plot(time_steps, out_plot, color='black', linewidth=3, label='Combined FPGA Output')

plt.title("Time-Domain Sine Waves (Hardware Output)", fontsize=14, fontweight='bold')
plt.xlabel("Time", fontsize=12)
plt.ylabel("Signal Amplitude", fontsize=12)
plt.grid(True, alpha=0.5)
plt.legend(loc='upper right')
plt.show()